In [26]:
# Imports
from langchain.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_openai import ChatOpenAI 
from langchain.prompts import ChatPromptTemplate 
import re 
import os 
from dotenv import load_dotenv 
import requests 

#load environment variables from .env
load_dotenv()

#read the api key for OpenWeather
api_key = os.getenv("OPENWEATHERMAP_API_KEY")
url = "https://api.openweathermap.org/data/2.5/forecast"


# Step 1: Define weather search function
@tool
def get_weather(city: str) -> str:
    """Get 5-day forecast for a given city in a single call. Return the daily max temperature and issue a travel
    advisory (Do not Travel/Safe to Travel).
    """
   
    try:
        api_call_params = {"q":city, "appid":api_key, "units":"metric"}
        Api_Response = requests.get(url,params=api_call_params)
        
        if Api_Response.status_code ==200:
            data = Api_Response.json()
            weather_forecasts = {}
            for entry in data["list"]:
                date = entry["dt_txt"].split()[0]
                temp = entry["main"]["temp"]
                weather_forecasts.setdefault(date,[]).append(temp)

            Weather_Advisory = []
            for date, temps in weather_forecasts.items():
                max_temp = max(temps)
                if max_temp > 40:
                    Advice = "Extreme Heat : Do not Travel"

                else:
                    Advice = "Safe to Travel"

                Weather_Advisory.append(f"{date}: {max_temp:.1f}.{Advice}")

            return ("\n".join(Weather_Advisory)+"\n")
            #return f"{'\n'.join(Weather_Advisory)}\n"
            
        else:
            print(f"API call failed with status {Api_Response.status_code}")

    except Exception as e:
        return (f"Error occurred: {str(e)}")     
   
       

# Step 2 : Create LLM

chat_model= ChatOpenAI(model="gpt-3.5-turbo",temperature =0)

# Step 3 : Prompt Template
prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant"),
    ("user", "{input}"),
    ("placeholder","{agent_scratchpad}")
])

agent = create_tool_calling_agent(
    llm=chat_model,
    tools=[get_weather],
    prompt = prompt 
)

agent_executor = AgentExecutor(agent=agent,tools=[get_weather], verbose=True,
                              return_intermediate_steps=False)

# Step 4: Invoke the agent
Agent_Output = agent_executor.invoke({
    "input" : "Give me the 5-day weather forecast for {city}, with daily max temperature and a travel advisory"
})

print(f"Please read my travel advisory below\n:{Agent_Output['output']}") 





> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'New York'}`


2026-07-30: 26.1.Safe to Travel
2026-07-31: 29.8.Safe to Travel
2026-08-01: 28.9.Safe to Travel
2026-08-02: 27.7.Safe to Travel
2026-08-03: 25.4.Safe to Travel
2026-08-04: 24.8.Safe to Travel
Here is the 5-day weather forecast for New York with the daily max temperature and travel advisory:

- July 30, 2026: 26.1°C, Safe to Travel
- July 31, 2026: 29.8°C, Safe to Travel
- August 1, 2026: 28.9°C, Safe to Travel
- August 2, 2026: 27.7°C, Safe to Travel
- August 3, 2026: 25.4°C, Safe to Travel

It is safe to travel to New York based on the weather forecast.

> Finished chain.
Please read my travel advisory below
:Here is the 5-day weather forecast for New York with the daily max temperature and travel advisory:

- July 30, 2026: 26.1°C, Safe to Travel
- July 31, 2026: 29.8°C, Safe to Travel
- August 1, 2026: 28.9°C, Safe to Travel
- August 2, 2026: 27.7°C, Safe to Travel
- August 3, 2026: 25.4°C